# kwargs-pass-through-recipe — faded example 3: Replay backward via **recipe.kwargs — fill in the splat call

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `kwargs-pass-through-recipe`. The last cell reports your progress on the `Backprop: Kwargs pass-through` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Kwargs pass-through` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`kwargs-pass-through-recipe`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "kwargs-pass-through-recipe"
DD_SUBTOPIC = "Backprop: Kwargs pass-through"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The stored `recipe.kwargs` allows the backward function to be invoked with the exact kwargs used at forward time, without the reverse-pass code needing to know which kwargs the op uses. This is done by splatting: `back_fn(grad_out, out, parent, **recipe.kwargs)`.

## Faded exercise 3

A `sum_back` function and a forward-computed `out_tensor` with a populated `recipe` are provided. Fill in the call to `sum_back` that uses `**recipe.kwargs` to recover the `axis` and `keepdims` values automatically.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import numpy as np
from dataclasses import dataclass
from typing import Any, Callable, Optional

@dataclass
class Recipe:
    func: Any
    args: tuple
    kwargs: dict
    parents: dict

class MiniTensor:
    def __init__(self, array):
        self.array = array
        self.recipe: Optional[Recipe] = None

def sum_back(grad_out, out, x, axis=None, keepdims=False):
    if not keepdims and axis is not None:
        grad_out = np.expand_dims(grad_out, axis=axis)
    return np.broadcast_to(grad_out, x.shape).copy()

def replay_sum_back(out_tensor: MiniTensor, grad_out: np.ndarray) -> np.ndarray:
    recipe = out_tensor.recipe
    parent_arr = recipe.parents[0].array
    return sum_back(grad_out, out_tensor.array, parent_arr, **recipe.kwargs)


def _test():
    import numpy as np

    x_arr = np.ones((3, 4))
    x = MiniTensor(x_arr)
    # Manually build a recipe for sum(axis=1)
    out_arr = x_arr.sum(axis=1)  # shape (3,)
    out = MiniTensor(out_arr)
    out.recipe = Recipe(np.sum, (x_arr,), {'axis': 1}, {0: x})

    grad_out = np.ones(3)
    grad_x = replay_sum_back(out, grad_out)
    assert grad_x.shape == (3, 4), f'expected (3,4) got {grad_x.shape}'
    np.testing.assert_allclose(grad_x, np.ones((3, 4)))


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import numpy as np
from dataclasses import dataclass
from typing import Any, Callable, Optional

@dataclass
class Recipe:
    func: Any
    args: tuple
    kwargs: dict
    parents: dict

class MiniTensor:
    def __init__(self, array):
        self.array = array
        self.recipe: Optional[Recipe] = None

def sum_back(grad_out, out, x, axis=None, keepdims=False):
    if not keepdims and axis is not None:
        grad_out = np.expand_dims(grad_out, axis=axis)
    return np.broadcast_to(grad_out, x.shape).copy()

def replay_sum_back(out_tensor: MiniTensor, grad_out: np.ndarray) -> np.ndarray:
    recipe = out_tensor.recipe
    parent_arr = recipe.parents[0].array
    return sum_back(grad_out, out_tensor.array, parent_arr, **recipe.kwargs)
```
</details>